# Divvy Bike-Share 2023 — Data Preparation

This notebook prepares the 2023 Divvy bike-share dataset for exploratory analysis and machine learning.

The raw monthly trip datasets are stored in AWS S3 and are loaded directly into Python for processing.

The preparation workflow is organized into the following stages:

1. Set up the Python environment
2. Connect to AWS S3 and load the raw Divvy datasets
3. Explore the combined dataset and examine data quality
4. Transform the data and create analytical features
5. Clean identified data-quality issues
6. Validate the cleaned dataset
7. Prepare and merge historical weather data
8. Export the processed datasets to AWS S3

## 1. Setup

- Import the Python libraries required for data ingestion, processing, and analysis.
- Import necessary libraries for importing the weather data for Chicago in 2023

In [ ]:
import boto3
import zipfile
import pandas as pd
from io import BytesIO
import gc
import os

In [2]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

## 2. AWS Configuration and Data Loading

- The original 2023 Divvy trip datasets are stored in the project's private AWS S3 bucket.

- This section establishes the AWS connection, identifies the monthly ZIP files stored in the raw-data prefix, loads each monthly dataset directly from S3, and combines them into a single DataFrame representing the full year.

### 2.1 Configure AWS Connection

- A local AWS CLI profile is used to authenticate with AWS. The raw and processed S3 locations are defined here for use throughout the notebook.

In [3]:
session = boto3.Session(profile_name="rami-main")

s3 = session.client(
    "s3",
    region_name="eu-central-1"
)

bucket_name = "divvy-bike-project-rami-865411539075-eu-central-1-an"

raw_prefix = "divvy/2023/raw/"
processed_prefix = "divvy/2023/processed/"

### 2.2 Locate Raw Divvy Files in S3

- The raw S3 prefix is inspected to identify the 12 monthly Divvy ZIP files for 2023.

In [4]:
response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix=raw_prefix
)

zip_keys = []

for item in response.get("Contents", []):
    key = item["Key"]

    if key.endswith(".zip"):
        zip_keys.append(key)

zip_keys.sort()

print(f"Number of ZIP files found: {len(zip_keys)}")

for key in zip_keys:
    print(key)

Number of ZIP files found: 12
divvy/2023/raw/202301-divvy-tripdata.zip
divvy/2023/raw/202302-divvy-tripdata.zip
divvy/2023/raw/202303-divvy-tripdata.zip
divvy/2023/raw/202304-divvy-tripdata.zip
divvy/2023/raw/202305-divvy-tripdata.zip
divvy/2023/raw/202306-divvy-tripdata.zip
divvy/2023/raw/202307-divvy-tripdata.zip
divvy/2023/raw/202308-divvy-tripdata.zip
divvy/2023/raw/202309-divvy-tripdata.zip
divvy/2023/raw/202310-divvy-tripdata.zip
divvy/2023/raw/202311-divvy-tripdata.zip
divvy/2023/raw/202312-divvy-tripdata.zip


### 2.3 Load the Monthly Divvy Datasets

- Each ZIP archive is retrieved directly from S3 and opened in memory. The CSV file inside each archive is loaded into a pandas DataFrame and stored for later combination.

- The source ZIP files remain unchanged in the raw S3 layer.

In [5]:
monthly_data = []

for key in zip_keys:
    response = s3.get_object(Bucket=bucket_name, Key=key)
    zip_data = response["Body"].read()

    with zipfile.ZipFile(BytesIO(zip_data), "r") as z:
        for csv_file in z.namelist():
            if csv_file.endswith(".csv") and "__MACOSX" not in csv_file:
                df_month = pd.read_csv(z.open(csv_file))
                monthly_data.append(df_month)
                print(key, df_month.shape)

divvy/2023/raw/202301-divvy-tripdata.zip (190301, 13)
divvy/2023/raw/202302-divvy-tripdata.zip (190445, 13)
divvy/2023/raw/202303-divvy-tripdata.zip (258678, 13)
divvy/2023/raw/202304-divvy-tripdata.zip (426590, 13)
divvy/2023/raw/202305-divvy-tripdata.zip (604827, 13)
divvy/2023/raw/202306-divvy-tripdata.zip (719618, 13)
divvy/2023/raw/202307-divvy-tripdata.zip (767650, 13)
divvy/2023/raw/202308-divvy-tripdata.zip (771693, 13)
divvy/2023/raw/202309-divvy-tripdata.zip (666371, 13)
divvy/2023/raw/202310-divvy-tripdata.zip (537113, 13)
divvy/2023/raw/202311-divvy-tripdata.zip (362518, 13)
divvy/2023/raw/202312-divvy-tripdata.zip (224073, 13)


### 2.4 Combine the Monthly Datasets

- The 12 monthly DataFrames are combined into one dataset representing all Divvy trips recorded in 2023.

In [6]:
df = pd.concat(monthly_data, ignore_index=True)

print(f"Combined dataset shape: {df.shape}")

Combined dataset shape: (5719877, 13)


In [7]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,F96D5A74A3E41399,electric_bike,2023-01-21 20:05:42,2023-01-21 20:16:33,Lincoln Ave & Fullerton Ave,TA1309000058,Hampden Ct & Diversey Ave,202480.0,41.924074,-87.646278,41.930000,-87.640000,member
1,13CB7EB698CEDB88,classic_bike,2023-01-10 15:37:36,2023-01-10 15:46:05,Kimbark Ave & 53rd St,TA1309000037,Greenwood Ave & 47th St,TA1308000002,41.799568,-87.594747,41.809835,-87.599383,member
2,BD88A2E670661CE5,electric_bike,2023-01-02 07:51:57,2023-01-02 08:05:11,Western Ave & Lunt Ave,RP-005,Valli Produce - Evanston Plaza,599,42.008571,-87.690483,42.039742,-87.699413,casual
3,C90792D034FED968,classic_bike,2023-01-22 10:52:58,2023-01-22 11:01:44,Kimbark Ave & 53rd St,TA1309000037,Greenwood Ave & 47th St,TA1308000002,41.799568,-87.594747,41.809835,-87.599383,member
4,3397017529188E8A,classic_bike,2023-01-12 13:58:01,2023-01-12 14:13:20,Kimbark Ave & 53rd St,TA1309000037,Greenwood Ave & 47th St,TA1308000002,41.799568,-87.594747,41.809835,-87.599383,member


### 2.5 Release Unused Objects from Memory


In [8]:
del monthly_data
del df_month
del zip_data

gc.collect()

144

## 3. Data Exploration and Quality Examination

- Before modifying or cleaning the combined dataset, the raw data is examined to understand its structure and identify potential quality issues.

- The examination covers dataset dimensions, identifiers, duplicate records, missing values, categorical variables, station information, geographic coordinates, timestamps, and ride durations.

- No records are removed during this stage. Cleaning decisions are made only after the identified issues have been investigated.

### 3.1 Dataset Overview

The overall structure of the combined dataset is reviewed, including its dimensions, column data types, missing values, and number of unique values.

In [9]:
print(f"Number of Rows: {df.shape[0]:,}")
print(f"Number of Columns: {df.shape[1]}")

Number of Rows: 5,719,877
Number of Columns: 13


In [10]:
# A quick data summary

data_summary = pd.DataFrame({"data_type": df.dtypes,
                             "missing_values": df.isna().sum(),
                             "missing_percentage": (df.isna().mean()* 100).round(2),
                             "number_of_unique_values": df.nunique()
})

data_summary

,data_type,missing_values,missing_percentage,number_of_unique_values
ride_id,object,0,0.00,5719877
rideable_type,object,0,0.00,3
started_at,object,0,0.00,4823909
ended_at,object,0,0.00,4835702
start_station_name,object,875716,15.31,1592
start_station_id,object,875848,15.31,1516
end_station_name,object,929202,16.25,1597
end_station_id,object,929343,16.25,1520
start_lat,float64,0,0.00,789702
start_lng,float64,0,0.00,748738


### 3.2 Ride ID and Duplicate Records

- `ride_id` is the unique identifier assigned to each trip. It is examined for missing and duplicated values.

- The dataset is also checked for completely duplicated rows.

In [11]:
print("Missing ride IDs:", df["ride_id"].isna().sum())
print("Duplicated ride IDs:", df["ride_id"].duplicated().sum())
print("Duplicated rows:", df.duplicated().sum())

Missing ride IDs: 0
Duplicated ride IDs: 0
Duplicated rows: 0


### 3.3 Missing Values

- Missing values are examined across all columns to determine where data is incomplete and which variables require further investigation.

In [12]:
df.isna().sum().sort_values(ascending=False)

end_station_id        929343
end_station_name      929202
start_station_id      875848
start_station_name    875716
end_lat                 6990
end_lng                 6990
started_at                 0
rideable_type              0
ride_id                    0
ended_at                   0
start_lat                  0
start_lng                  0
member_casual              0
dtype: int64

### 3.4 Ride Type

- The available bike types are examined for missing values and unexpected categories.

In [13]:
df["rideable_type"].value_counts(dropna=False)

rideable_type
electric_bike    2945579
classic_bike     2696011
docked_bike        78287
Name: count, dtype: int64

### 3.5 Membership Type

- The rider membership categories are examined for missing values and unexpected categories.

In [14]:
df["member_casual"].value_counts(dropna=False)

member_casual
member    3660698
casual    2059179
Name: count, dtype: int64

### 3.6 Station Information

- Start and end station fields are examined for missing values and inconsistencies.

- Missing station information is also compared across bike types to determine whether missingness is concentrated within a particular type of bike.

In [15]:
station_columns = [
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id"
]

df[station_columns].isna().sum()

start_station_name    875716
start_station_id      875848
end_station_name      929202
end_station_id        929343
dtype: int64

In [16]:
df.groupby("rideable_type").agg(
    total_rides=("rideable_type", "size"),
    missing_start_station=("start_station_name", lambda x: x.isna().sum()),
    missing_end_station=("end_station_name", lambda x: x.isna().sum())
)

,total_rides,missing_start_station,missing_end_station
rideable_type,,,
classic_bike,2696011,34,5116
docked_bike,78287,0,2047
electric_bike,2945579,875682,922039


### 3.7 Geographic Coordinates

- Start and end coordinates are examined for missing values, invalid zero values, and unusual latitude or longitude ranges.

In [17]:
coordinate_columns = [
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng"
]

df[coordinate_columns].isna().sum()

start_lat       0
start_lng       0
end_lat      6990
end_lng      6990
dtype: int64

In [18]:
print("Start latitude range:", df["start_lat"].min(), "-", df["start_lat"].max())
print("Start longitude range:", df["start_lng"].min(), "-", df["start_lng"].max())
print("End latitude range:", df["end_lat"].min(), "-", df["end_lat"].max())
print("End longitude range:", df["end_lng"].min(), "-", df["end_lng"].max())

print()
print("Zero start latitude:", (df["start_lat"] == 0).sum())
print("Zero start longitude:", (df["start_lng"] == 0).sum())
print("Zero end latitude:", (df["end_lat"] == 0).sum())
print("Zero end longitude:", (df["end_lng"] == 0).sum())

Start latitude range: 41.63 - 42.07
Start longitude range: -87.94 - -87.46
End latitude range: 0.0 - 42.18
End longitude range: -88.16 - 0.0

Zero start latitude: 0
Zero start longitude: 0
Zero end latitude: 3
Zero end longitude: 3


### 3.8 Timestamp and Ride Duration Quality

- The trip start and end timestamps are examined to determine whether they can be parsed correctly and whether the resulting ride durations contain invalid or extreme observations.

- Temporary datetime and duration variables are used for this examination so that the raw DataFrame remains unchanged during the quality-assessment stage.

In [19]:
started_at_check = pd.to_datetime(df["started_at"], format="mixed", errors="coerce")
ended_at_check = pd.to_datetime(df["ended_at"], format="mixed", errors="coerce")

print("Invalid start timestamps:", started_at_check.isna().sum())
print("Invalid end timestamps:", ended_at_check.isna().sum())

Invalid start timestamps: 0
Invalid end timestamps: 0


In [20]:
ride_duration_check = (ended_at_check - started_at_check).dt.total_seconds() / 60
ride_duration_check.describe()

count    5.719877e+06
mean     1.816932e+01
std      1.808430e+02
min     -1.665652e+04
25%      5.416667e+00
50%      9.533333e+00
75%      1.691667e+01
max      9.848907e+04
dtype: float64

In [21]:
ride_duration_check.quantile([
    0,
    0.01,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    0.999,
    1
])

0.000   -16656.516667
0.010        0.266667
0.250        5.416667
0.500        9.533333
0.750       16.916667
0.900       29.166667
0.950       41.333333
0.990       98.766667
0.999     1499.633333
1.000    98489.066667
dtype: float64

In [22]:
print("Negative rides:", (ride_duration_check < 0).sum())
print("Zero-minute rides:", (ride_duration_check == 0).sum())
print("less than half a minute ride:", (ride_duration_check < 0.5).sum())
print("Rides under 1 minute:", (ride_duration_check < 1).sum())
print("Rides under 2 minutes:", (ride_duration_check < 2).sum())
print("Rides over 1 hour", (ride_duration_check > 60).sum())
print("Rides over 3 hours:", (ride_duration_check > 180).sum())
print("Rides over 6 hours:", (ride_duration_check > 360).sum())
print("Rides over 12 hours:", (ride_duration_check > 720).sum())
print("Rides over 24 hours:", (ride_duration_check > 1440).sum())

Negative rides: 272
Zero-minute rides: 997
less than half a minute ride: 95566
Rides under 1 minute: 149615
Rides under 2 minutes: 263293
Rides over 1 hour 140956
Rides over 3 hours: 21125
Rides over 6 hours: 12485
Rides over 12 hours: 9366
Rides over 24 hours: 6418


### 3.9 Investigation of Unusual Ride Durations

- Short and unusually long rides are examined in more detail before defining the final cleaning rules.

In [23]:
short_rides = ((ride_duration_check > 0) & (ride_duration_check< 1))

print("Rides more than Zero and under One minute: ", short_rides.sum())

Rides more than Zero and under One minute:  148346


In [24]:
short_rides_df = df.loc[(short_rides == True)]

In [25]:
same_station = (
    short_rides_df["start_station_id"].notna()
    & short_rides_df["end_station_id"].notna()
    & (short_rides_df["start_station_id"] == short_rides_df["end_station_id"])
)

different_station = (
    short_rides_df["start_station_id"].notna()
    & short_rides_df["end_station_id"].notna()
    & (short_rides_df["start_station_id"] != short_rides_df["end_station_id"])
)

missing_station = (
    short_rides_df["start_station_id"].isna()
    | short_rides_df["end_station_id"].isna()
)

print("More than Zero & under 1 minute - same station:", same_station.sum())
print("More than Zero & under 1 minute - different station:", different_station.sum())
print("More than Zero & under 1 minute - missing station info:", missing_station.sum())

More than Zero & under 1 minute - same station: 83519
More than Zero & under 1 minute - different station: 3430
More than Zero & under 1 minute - missing station info: 61397


In [26]:
rides_over_12h = df.loc[(ride_duration_check > 720) == True]

rides_over_12h["rideable_type"].value_counts()

rideable_type
classic_bike    7123
docked_bike     2243
Name: count, dtype: int64

In [27]:
print("Over 12 hours", rides_over_12h.shape[0])

Over 12 hours 9366


### 3.10 Data Quality Summary

- The initial examination identified the following findings:

    - No missing or duplicated ride IDs were found.
    - No completely duplicated records were found.
    - Ride type and membership type contain no missing or unexpected categories.
    - Missing station information is substantial and is concentrated primarily among electric bike trips.
    - Start coordinates are complete, while a small number of trips are missing end coordinates.
    - All start and end timestamps can be parsed successfully.
    - Ride duration contains negative, zero, and extremely long observations that require further consideration during cleaning.

- These findings will guide the cleaning decisions applied later in the notebook.

## 4. Data Transformation and Feature Engineering

- After examining the raw data, the dataset is transformed into a structure suitable for analysis.

- Timestamp columns are converted to datetime format, ride duration is calculated, and additional temporal and trip-level features are created for use in exploratory analysis and machine learning.

### 4.1 Convert Datetime Columns

- The trip start and end timestamps are converted from text to pandas datetime values so that time-based calculations and features can be created.

In [28]:
df['started_at'] = pd.to_datetime(df['started_at'], format = "mixed")
df['ended_at'] = pd.to_datetime(df['ended_at'], format = "mixed")

### 4.2 Create Ride Duration

- Ride duration is calculated as the difference between the trip end time and start time and is expressed in minutes.

In [29]:
df["ride_duration_min"] = ( df["ended_at"] - df["started_at"]).dt.total_seconds() / 60

### 4.3 Create Date and Time Features

- Creating new columns, such as: start_date, start_hour, day_of_week, is_weekend, month_name and season

In [30]:
df["start_date"] = df["started_at"].dt.normalize()
df["start_hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.day_name()
df['is_weekend'] = df["day_of_week"].apply(lambda x: 1 if x in ["Saturday","Sunday"] else 0)
df["month"] = df["started_at"].dt.month
df["month_name"] = df["started_at"].dt.month_name()

### 4.4 Create Season

- Each ride is assigned to a season based on the month in which the trip started.

In [31]:
def create_season(x):
    if x in ['March', 'April', 'May']:
        return "Spring"
    elif x in ['June', 'July', 'August']:
        return "Summer"
    elif x in ['September', 'October', 'November']:
            return "Fall"
    else:
         return "Winter"

df["season"] = df['month_name'].apply(create_season)

In [32]:
df.head(2)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lng,member_casual,ride_duration_min,start_date,start_hour,day_of_week,is_weekend,month,month_name,season
0,F96D5A74A3E41399,electric_bike,2023-01-21 20:05:42,2023-01-21 20:16:33,Lincoln Ave & Fullerton Ave,TA1309000058,Hampden Ct & Diversey Ave,202480.0,41.924074,-87.646278,...,-87.640000,member,10.850000,2023-01-21,20,Saturday,1,1,January,Winter
1,13CB7EB698CEDB88,classic_bike,2023-01-10 15:37:36,2023-01-10 15:46:05,Kimbark Ave & 53rd St,TA1309000037,Greenwood Ave & 47th St,TA1308000002,41.799568,-87.594747,...,-87.599383,member,8.483333,2023-01-10,15,Tuesday,0,1,January,Winter


## 5. Data Cleaning

- The data-quality examination identified a small number of invalid or unusual ride durations, missing station information and missing starting and endeing locations.

### 5.1 Ride Duration Cleaning

- The data-quality examination identified a small number of invalid or unusual ride durations.

- The following cleaning rules are applied:

    - Rides with a duration less than or equal to zero are removed.
    - Rides shorter than one minute are removed only when the start and end station are the same.
    - Rides between one minute and 12 hours are kept.
    - Rides longer than 12 hours (720 minutes) are treated as extreme duration outliers and removed for this analysis. These observations may represent unusually long rentals, unclosed trips, or recording anomalies.

In [33]:
df["ride_duration_min"].max()

np.float64(98489.06666666667)

In [34]:
df[df["ride_duration_min"] > 720].shape[0]

9366

In [35]:
df_clean = df[df["ride_duration_min"] > 0]

short_rides_same_station = df[(df["ride_duration_min"] > 0) &
                               (df["ride_duration_min"] <1) &
                               (df["start_station_id"] == df["end_station_id"])]

long_rides = df[df["ride_duration_min"] > 720]

df_clean = df_clean.drop(short_rides_same_station.index)
df_clean = df_clean.drop(long_rides.index)

df_clean.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lng,member_casual,ride_duration_min,start_date,start_hour,day_of_week,is_weekend,month,month_name,season
0,F96D5A74A3E41399,electric_bike,2023-01-21 20:05:42,2023-01-21 20:16:33,Lincoln Ave & Fullerton Ave,TA1309000058,Hampden Ct & Diversey Ave,202480.0,41.924074,-87.646278,...,-87.640000,member,10.850000,2023-01-21,20,Saturday,1,1,January,Winter
1,13CB7EB698CEDB88,classic_bike,2023-01-10 15:37:36,2023-01-10 15:46:05,Kimbark Ave & 53rd St,TA1309000037,Greenwood Ave & 47th St,TA1308000002,41.799568,-87.594747,...,-87.599383,member,8.483333,2023-01-10,15,Tuesday,0,1,January,Winter
2,BD88A2E670661CE5,electric_bike,2023-01-02 07:51:57,2023-01-02 08:05:11,Western Ave & Lunt Ave,RP-005,Valli Produce - Evanston Plaza,599,42.008571,-87.690483,...,-87.699413,casual,13.233333,2023-01-02,7,Monday,0,1,January,Winter
3,C90792D034FED968,classic_bike,2023-01-22 10:52:58,2023-01-22 11:01:44,Kimbark Ave & 53rd St,TA1309000037,Greenwood Ave & 47th St,TA1308000002,41.799568,-87.594747,...,-87.599383,member,8.766667,2023-01-22,10,Sunday,1,1,January,Winter
4,3397017529188E8A,classic_bike,2023-01-12 13:58:01,2023-01-12 14:13:20,Kimbark Ave & 53rd St,TA1309000037,Greenwood Ave & 47th St,TA1308000002,41.799568,-87.594747,...,-87.599383,member,15.316667,2023-01-12,13,Thursday,0,1,January,Winter


In [36]:
print(f"Original rows: {len(df):,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Rows removed: {len(df) - len(df_clean):,}")
print(f"Percentage removed: {(len(df) - len(df_clean)) / len(df) * 100:.2f}%")

Original rows: 5,719,877
Cleaned rows: 5,625,723
Rows removed: 94,154
Percentage removed: 1.65%


### 5.2 Geographic Coordinate Cleaning

- The coordinate quality check identified a small number of missing end coordinates and a few zero-valued end coordinates.

    - Rows with missing coordinates are retained because they are still useful for non-geographic analyses such as membership, bike type, temporal patterns, weather analysis, and ride-count prediction.

    - Zero-valued coordinates are invalid geographic values and are therefore converted to missing values rather than removing the entire trip.

In [37]:
print("Missing start latitude:", df_clean["start_lat"].isna().sum())
print("Missing start longitude:", df_clean["start_lng"].isna().sum())
print("Missing end latitude:", df_clean["end_lat"].isna().sum())
print("Missing end longitude:", df_clean["end_lng"].isna().sum())

print("\n\n")

print("Zero start latitude:", (df_clean["start_lat"] == 0).sum())
print("Zero start longitude:", (df_clean["start_lng"] == 0).sum())
print("Zero end latitude:", (df_clean["end_lat"] == 0).sum())
print("Zero end longitude:", (df_clean["end_lng"] == 0).sum())

Missing start latitude: 0
Missing start longitude: 0
Missing end latitude: 511
Missing end longitude: 511



Zero start latitude: 0
Zero start longitude: 0
Zero end latitude: 2
Zero end longitude: 2


In [38]:
df_clean.loc[df_clean["start_lat"] == 0, "start_lat"] = pd.NA
df_clean.loc[df_clean["start_lng"] == 0, "start_lng"] = pd.NA
df_clean.loc[df_clean["end_lat"] == 0, "end_lat"] = pd.NA
df_clean.loc[df_clean["end_lng"] == 0, "end_lng"] = pd.NA

In [39]:
print("Zero start latitude:", (df_clean["start_lat"] == 0).sum())
print("Zero start longitude:", (df_clean["start_lng"] == 0).sum())
print("Zero end latitude:", (df_clean["end_lat"] == 0).sum())
print("Zero end longitude:", (df_clean["end_lng"] == 0).sum())

Zero start latitude: 0
Zero start longitude: 0
Zero end latitude: 0
Zero end longitude: 0


### 5.3 Missing Station Information

- Missing station information is kept.

- The data-quality examination showed that missing station data is mostly among electric bike trips. Removing all records with missing station information would mess up with the propotions of the bike type used in the rides, that's why these records won't be removed.

### 5.4 Cleaning Summary

The cleaning process focused only on records for which there was sufficient evidence of a data-quality problem.

The following changes were applied:

- Rides with zero or negative duration were removed.
- Rides shorter than one minute were removed when they started and ended at the same known station.
- Rides longer than twelve hours were removed.
- Missing station information was kept to avoid disproportionately removing electric-bike trips.
- Missing geographic coordinates were retained because the trips remain useful for non-geographic analyses.
- Zero-valued geographic coordinates were converted to missing values.

The majority of the original dataset was preserved.

## 6. Validation of the Cleaned Dataset

- The cleaned dataset is validated to confirm that the intended cleaning rules were applied successfully.

In [40]:
print(
    "Zero or negative rides:",
    (df_clean["ride_duration_min"] <= 0).sum()
)

print(
    "Rides over 12 hours:",
    (df_clean["ride_duration_min"] > 720).sum()
)

short_same_station_remaining = (
    (df_clean["ride_duration_min"] > 0)
    & (df_clean["ride_duration_min"] < 1)
    & (df_clean["start_station_id"] == df_clean["end_station_id"])
)

print(
    "Short same-station rides:",
    short_same_station_remaining.sum()
)

Zero or negative rides: 0
Rides over 12 hours: 0
Short same-station rides: 0


### 6.1 Validation Summary

The validation checks confirm that the cleaning rules were applied successfully:

- No zero or negative ride durations remain.
- No rides longer than 12 hours remain.
- No rides shorter than one minute remain when the trip starts and ends at the same known station.

The cleaned Divvy dataset is now ready for enrichment with weather data.

### 6.2 Finalize the Cleaned Dataset

- The `ride_id` column was used to validate trip uniqueness during the data-quality examination.

- After confirming that no missing or duplicated ride IDs exist, the identifier is removed because it is not required for the subsequent analysis or machine-learning workflow.

In [41]:
df_clean.drop(columns="ride_id", inplace=True)

### 6.3 Remove the original dataset and temporary objects

In [42]:
# Remove the original dataset and temporary objects
del df

del started_at_check
del ended_at_check
del ride_duration_check

del short_rides
del short_rides_df
del same_station
del different_station
del missing_station
del rides_over_12h

del short_rides_same_station
del long_rides

gc.collect()

22

## 7. Weather Data Preparation and Merge

- Historical daily weather data for Chicago is retrieved from the Open-Meteo Historical Weather API for the full year of 2023.

- Weather information is added to the trip dataset to support later analysis of how temperature, precipitation, snowfall, and wind conditions relate to Divvy ridership.

### 7.1 Retrieve Historical Weather Data

- Daily weather observations are requested for Chicago from January 1 through December 31, 2023.

- The requested variables include:

    - Mean, minimum, and maximum temperature
    - Total precipitation
    - Total rainfall
    - Total snowfall
    - Maximum wind speed

In [43]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 41.85,
	"longitude": -87.65,
	"start_date": "2023-01-01",
	"end_date": "2023-12-31",
	"daily": ["temperature_2m_mean", "temperature_2m_min", "temperature_2m_max", "precipitation_sum", "rain_sum", "snowfall_sum", "wind_speed_10m_max"],
	"timezone": "America/Chicago",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(2).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(3).ValuesAsNumpy()
daily_rain_sum = daily.Variables(4).ValuesAsNumpy()
daily_snowfall_sum = daily.Variables(5).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(6).ValuesAsNumpy()

daily_data = {
    "start_date": pd.date_range(
        start=params["start_date"],
        end=params["end_date"],
        freq="D"
    )
}

Coordinates: 41.8629150390625°N -87.64877319335938°E
Elevation: 179.0 m asl
Timezone: b'America/Chicago'b'GMT-5'
Timezone difference to GMT+0: -18000s


### 7.2 Create the Weather DataFrame

- Create dataframe of the daily weather in Chicago from the produced parameters

In [44]:
daily_data["temperature_mean_c"] = daily_temperature_2m_mean
daily_data["temperature_min_c"] = daily_temperature_2m_min
daily_data["temperature_max_c"] = daily_temperature_2m_max
daily_data["precipitation_mm"] = daily_precipitation_sum
daily_data["rain_mm"] = daily_rain_sum
daily_data["snowfall_cm"] = daily_snowfall_sum
daily_data["wind_speed_max_kmh"] = daily_wind_speed_10m_max

weather_df = pd.DataFrame(data = daily_data)
weather_df.head()

,start_date,temperature_mean_c,temperature_min_c,temperature_max_c,precipitation_mm,rain_mm,snowfall_cm,wind_speed_max_kmh
0,2023-01-01,4.122917,1.20,8.50,3.300000,3.300000,0.00,17.068707
1,2023-01-02,2.052083,-0.45,4.55,0.300000,0.300000,0.00,18.075441
2,2023-01-03,7.339583,3.95,12.70,13.100001,13.100001,0.00,23.598541
3,2023-01-04,2.610417,0.15,8.80,0.000000,0.000000,0.00,23.006226
4,2023-01-05,0.152083,-0.40,0.90,1.300000,0.100000,0.91,21.313093


### 7.3 Validate the Weather Data

- Before merging anything, inspect it:

In [45]:
weather_df.dtypes

start_date            datetime64[ns]
temperature_mean_c           float32
temperature_min_c            float32
temperature_max_c            float32
precipitation_mm             float32
rain_mm                      float32
snowfall_cm                  float32
wind_speed_max_kmh           float32
dtype: object

In [46]:
print("Weather dataset shape:", weather_df.shape)

print()
print("Date range:")
print(weather_df["start_date"].min())
print(weather_df["start_date"].max())

print()
print("Missing values:")
print(weather_df.isna().sum())

Weather dataset shape: (365, 8)

Date range:
2023-01-01 00:00:00
2023-12-31 00:00:00

Missing values:
start_date            0
temperature_mean_c    0
temperature_min_c     0
temperature_max_c     0
precipitation_mm      0
rain_mm               0
snowfall_cm           0
wind_speed_max_kmh    0
dtype: int64


### 7.4 Merge Weather with Divvy Trip Data

- The daily weather dataset is merged with the cleaned Divvy dataset using the trip start date.

- A left join is used so that all cleaned Divvy trips are preserved even if weather information is unavailable for a particular date.

In [47]:
df_clean = df_clean.merge(weather_df, on="start_date", how="left")
df_clean.head()

,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,...,month,month_name,season,temperature_mean_c,temperature_min_c,temperature_max_c,precipitation_mm,rain_mm,snowfall_cm,wind_speed_max_kmh
0,electric_bike,2023-01-21 20:05:42,2023-01-21 20:16:33,Lincoln Ave & Fullerton Ave,TA1309000058,Hampden Ct & Diversey Ave,202480.0,41.924074,-87.646278,41.930000,...,1,January,Winter,-0.791667,-2.80,2.50,0.0,0.0,0.00,16.055355
1,classic_bike,2023-01-10 15:37:36,2023-01-10 15:46:05,Kimbark Ave & 53rd St,TA1309000037,Greenwood Ave & 47th St,TA1308000002,41.799568,-87.594747,41.809835,...,1,January,Winter,3.327083,0.65,8.00,0.4,0.4,0.00,20.124611
2,electric_bike,2023-01-02 07:51:57,2023-01-02 08:05:11,Western Ave & Lunt Ave,RP-005,Valli Produce - Evanston Plaza,599,42.008571,-87.690483,42.039742,...,1,January,Winter,2.052083,-0.45,4.55,0.3,0.3,0.00,18.075441
3,classic_bike,2023-01-22 10:52:58,2023-01-22 11:01:44,Kimbark Ave & 53rd St,TA1309000037,Greenwood Ave & 47th St,TA1308000002,41.799568,-87.594747,41.809835,...,1,January,Winter,-0.608333,-1.35,0.75,2.5,0.0,1.75,19.201874
4,classic_bike,2023-01-12 13:58:01,2023-01-12 14:13:20,Kimbark Ave & 53rd St,TA1309000037,Greenwood Ave & 47th St,TA1308000002,41.799568,-87.594747,41.809835,...,1,January,Winter,2.943750,1.80,6.65,0.1,0.1,0.00,37.883907


### 7.5 Validate the merge

- Cheking the shape of the new data & if any of the new imported values is null

In [48]:
print("Dataset shape after weather merge:")
print(df_clean.shape)

Dataset shape after weather merge:
(5625723, 27)


In [49]:
weather_columns = [
    "temperature_mean_c",
    "temperature_min_c",
    "temperature_max_c",
    "precipitation_mm",
    "rain_mm",
    "snowfall_cm",
    "wind_speed_max_kmh"
]

df_clean[weather_columns].isna().sum()

temperature_mean_c    0
temperature_min_c     0
temperature_max_c     0
precipitation_mm      0
rain_mm               0
snowfall_cm           0
wind_speed_max_kmh    0
dtype: int64

### 7.6 Release Weather Objects from Memory

In [50]:
del weather_df
del daily_data

gc.collect()

0

In [51]:
df_clean.shape

(5625723, 27)

## 8. Export Processed Data to S3

- The final cleaned and weather-enriched Divvy dataset is exported to the processed layer of the project's AWS S3 bucket.

- The dataset is stored in Parquet format because it provides efficient storage, preserves column data types, and supports faster loading for the exploratory analysis and machine-learning notebooks.

### 8.1 Final Dataset Check

- Before exporting the processed dataset, its final dimensions, columns, and memory usage are reviewed.

In [52]:
print(f"Final dataset shape: {df_clean.shape}")

print()
print("Final columns:")
print(df_clean.columns.tolist())

print()
print(
    f"Memory usage: "
    f"{df_clean.memory_usage(deep=True).sum() / 1024**3:.2f} GB")


Final dataset shape: (5625723, 27)

Final columns:
['rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'ride_duration_min', 'start_date', 'start_hour', 'day_of_week', 'is_weekend', 'month', 'month_name', 'season', 'temperature_mean_c', 'temperature_min_c', 'temperature_max_c', 'precipitation_mm', 'rain_mm', 'snowfall_cm', 'wind_speed_max_kmh']

Memory usage: 3.65 GB


### 8.2 Save the Processed Dataset as Parquet

- The processed dataset is first saved locally in Parquet format using Snappy compression.

- Parquet significantly reduces the storage size of the dataset while preserving its column data types.

In [53]:
local_file = "divvy_2023_processed.parquet"

df_clean.to_parquet(
    local_file,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

In [54]:
local_file_size = os.path.getsize(local_file)

print(
    f"Parquet file size: "
    f"{local_file_size / 1024**3:.2f} GB"
)

Parquet file size: 0.21 GB


### 8.3 Upload the Processed Dataset to S3

- The Parquet file is uploaded to the processed prefix of the project's private AWS S3 bucket.

In [55]:
processed_key = (
    processed_prefix
    + "divvy_2023_processed.parquet"
)

s3.upload_file(
    local_file,
    bucket_name,
    processed_key
)

print("Processed dataset uploaded successfully.")
print(f"S3 location: s3://{bucket_name}/{processed_key}")

Processed dataset uploaded successfully.
S3 location: s3://divvy-bike-project-rami-865411539075-eu-central-1-an/divvy/2023/processed/divvy_2023_processed.parquet


### 8.4 Validate the S3 Upload

- The uploaded S3 object is checked to confirm that the processed dataset was successfully stored and that its size matches the local Parquet file.

In [56]:
processed_object = s3.head_object(
    Bucket=bucket_name,
    Key=processed_key
)

s3_file_size = processed_object["ContentLength"]

print(
    f"Local file size: "
    f"{local_file_size / 1024**2:.2f} MB"
)

print(
    f"S3 file size: "
    f"{s3_file_size / 1024**2:.2f} MB"
)

print(
    "File sizes match:",
    local_file_size == s3_file_size
)

Local file size: 210.73 MB
S3 file size: 210.73 MB
File sizes match: True


### 8.5 Delete local temporary file

In [57]:
os.remove(local_file)

print("Temporary local Parquet file removed.")

Temporary local Parquet file removed.


### 8.6 Final Processing Summary

The processed 2023 Divvy dataset has been successfully prepared and stored in the project's processed AWS S3 layer.

The final dataset contains:

- Cleaned Divvy trip records
- Ride-duration information
- Date and time features
- Weekend and seasonal features
- Station and geographic information where available
- Daily Chicago temperature data
- Daily precipitation, rainfall, and snowfall data
- Daily maximum wind speed

The processed Parquet dataset can now be used directly by the exploratory analysis and machine-learning notebooks without repeating the raw-data preparation workflow.

## Data Preparation Complete

The 2023 Divvy trip dataset has now been:

1. Loaded from the raw AWS S3 layer
2. Examined for data-quality issues
3. Transformed and enriched with analytical features
4. Cleaned using the defined data-quality rules
5. Validated after cleaning
6. Enriched with historical Chicago weather data
7. Exported to the processed AWS S3 layer in Parquet format

The processed dataset is now ready for exploratory data analysis and machine learning.